# Quant Finance + AI Learning Curve


> Disclaimer: This is educational research code, not investment advice.


## 0. Setup

We import libraries, set a random seed for reproducibility, and define small utility helpers used throughout the notebook.

In [2]:
# If you run this notebook in a fresh environment, you may need:
# !pip -q install yfinance scikit-learn

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd

import yfinance as yf

from dataclasses import dataclass
from typing import Dict, Tuple, List

# Reproducibility
SEED = 42
np.random.seed(SEED)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

def sharpe_ratio(returns: pd.Series, periods_per_year: int = 252) -> float:
    """Annualized Sharpe ratio (no risk-free rate for simplicity)."""
    returns = returns.dropna()
    if returns.std(ddof=0) == 0 or len(returns) == 0:
        return np.nan
    return (returns.mean() / returns.std(ddof=0)) * math.sqrt(periods_per_year)

def max_drawdown(equity_curve: pd.Series) -> float:
    """Max drawdown computed from an equity curve."""
    ec = equity_curve.dropna()
    if len(ec) == 0:
        return np.nan
    peak = ec.cummax()
    dd = (ec / peak) - 1.0
    return dd.min()

def print_performance(name: str, daily_returns: pd.Series) -> None:
    """Convenience function to print common backtest metrics."""
    daily_returns = daily_returns.dropna()
    equity = (1 + daily_returns).cumprod()
    cagr = equity.iloc[-1] ** (252 / len(daily_returns)) - 1 if len(daily_returns) > 0 else np.nan
    print(f"\n{name}")
    print("-" * len(name))
    print(f"CAGR:        {cagr:8.2%}")
    print(f"Sharpe:      {sharpe_ratio(daily_returns):8.2f}")
    print(f"Max drawdown:{max_drawdown(equity):8.2%}")
    print(f"Days:        {len(daily_returns):8d}")

## 1. Download data (public, reproducible)

We will work with a liquid benchmark ETF (**SPY**) and keep the dataset small and accessible. You can swap the ticker to anything supported by Yahoo Finance.

In [3]:
TICKER = "SPY"
START = "2010-01-01"
END = None  # None means 'up to latest available'

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
raw = raw.rename(columns=str.lower)

# We mainly need close and volume; keep others for indicators.
raw.tail()

Price,close,high,low,open,volume
Ticker,spy,spy,spy,spy,spy
Date,,,,,
2026-01-16,691.659973,694.250000,690.099976,693.659973,79289200
2026-01-20,677.580017,684.770020,676.570007,681.489990,111623300
2026-01-21,685.400024,688.739990,678.130005,679.650024,127844500
2026-01-22,688.979980,691.130005,686.919983,689.849976,77112200
2026-01-23,689.229980,690.960022,687.159973,688.150024,63016300


## 2. Create a clean return series

Quant work usually starts with returns. We'll build:
- daily log return
- next-day forward return (used as a target)

We also keep a simple 'direction' label for classification tasks.

In [4]:
df = raw.copy()

# Use simple returns for backtesting (equity curve = cumprod(1 + returns))
df["ret"] = df["close"].pct_change()
df["fwd_ret_1d"] = df["ret"].shift(-1)
df["up_1d"] = (df["fwd_ret_1d"] > 0).astype(int)

df[["close", "ret", "fwd_ret_1d", "up_1d"]].dropna().head()

Price,close,ret,fwd_ret_1d,up_1d
Ticker,spy,,,
Date,,,,
2010-01-05,85.253052,0.002647,0.000704,1
2010-01-06,85.313095,0.000704,0.004221,1
2010-01-07,85.673180,0.004221,0.003328,1
2010-01-08,85.958290,0.003328,0.001397,1
2010-01-11,86.078339,0.001397,-0.009326,0


## 3. Feature engineering (technical indicators)

We create a small set of **interpretable features**:
- moving-average gaps
- rolling volatility
- momentum
- RSI

These are easy to compute with pandas and require no external datasets.

In [5]:
def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    # Wilder's RSI uses exponential moving averages; here we use simple rolling means for clarity.
    roll_up = up.rolling(window).mean()
    roll_down = down.rolling(window).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Moving averages
    for w in [5, 10, 20, 50, 200]:
        out[f"sma_{w}"] = out["close"].rolling(w).mean()
    out["sma_fast_gap"] = (out["sma_10"] / out["sma_50"]) - 1.0
    out["sma_slow_gap"] = (out["sma_50"] / out["sma_200"]) - 1.0

    # Momentum (lookback returns)
    for w in [5, 10, 20, 60]:
        out[f"mom_{w}"] = np.log(out["close"]).diff(w)

    # Volatility
    for w in [5, 20, 60]:
        out[f"vol_{w}"] = out["ret"].rolling(w).std(ddof=0)

    # Volume features
    out["vol_z_20"] = (out["volume"] - out["volume"].rolling(20).mean()) / out["volume"].rolling(20).std(ddof=0)

    # RSI
    out["rsi_14"] = rsi(out["close"], 14)

    return out

df_feat = add_features(df)

feature_cols = [
    "sma_fast_gap", "sma_slow_gap",
    "mom_5", "mom_10", "mom_20", "mom_60",
    "vol_5", "vol_20", "vol_60",
    "vol_z_20",
    "rsi_14",
]

df_ml = df_feat.dropna().copy()
df_ml[feature_cols + ["fwd_ret_1d", "up_1d"]].head()

Price,sma_fast_gap,sma_slow_gap,mom_5,mom_10,mom_20,mom_60,vol_5,vol_20,vol_60,vol_z_20,rsi_14,fwd_ret_1d,up_1d
Ticker,,,,,,,,,,,,,
Date,,,,,,,,,,,,,
2010-10-18,0.048092,0.002314,0.013877,0.039052,0.035016,0.074193,0.003869,0.007516,0.009727,-1.130253,73.656935,-0.013104,0
2010-10-19,0.047895,0.002869,-0.002396,0.005929,0.023841,0.050641,0.007414,0.008157,0.009824,1.967673,62.584139,0.009766,1
2010-10-20,0.048399,0.003699,-0.000424,0.015734,0.038485,0.060449,0.007884,0.008232,0.009889,0.176353,69.121072,0.002206,1
2010-10-21,0.048637,0.005117,0.005688,0.019144,0.048832,0.069128,0.007663,0.007901,0.009841,0.625283,68.410615,0.001862,1
2010-10-22,0.048325,0.006705,0.005508,0.015412,0.030281,0.075872,0.007659,0.006716,0.009810,-1.626044,75.785368,0.002957,1


## 4. Baseline: buy and hold

Before ML, we set a baseline. For SPY, buy and hold is a strong benchmark.

We use daily log returns and convert them into an equity curve.

In [6]:
buy_hold = df_ml["ret"]
print_performance("Buy & Hold (SPY)", buy_hold)


Buy & Hold (SPY)
----------------
CAGR:          14.33%
Sharpe:          0.87
Max drawdown: -33.72%
Days:            3839


## 5. Baseline strategy: moving-average crossover

A classic quant strategy:
- Go long when SMA(10) > SMA(50)
- Go to cash when SMA(10) ≤ SMA(50)

This sets a second baseline that uses *only* price history (same information as ML).

In [7]:
signal_ma = (df_ml["sma_10"] > df_ml["sma_50"]).astype(int)  # 1 = long, 0 = cash
# Trade at next open is ideal; for simplicity we apply signal to next day's return
strategy_ma = signal_ma.shift(1).fillna(0) * df_ml["ret"]

print_performance("MA Crossover (10/50)", strategy_ma)


MA Crossover (10/50)
--------------------
CAGR:           7.73%
Sharpe:          0.73
Max drawdown: -20.66%
Days:            3839


## 6. Machine learning framing: predict next-day direction

Now we turn the same problem into a supervised ML task:
- Features: technical indicators
- Target: next-day direction `up_1d`

We will use a **time-based split** (no shuffling) to avoid look-ahead bias.

In [8]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = df_ml[feature_cols]
y = df_ml["up_1d"]

# Train/test split by date (simple holdout)
split_date = "2018-01-01"
X_train, X_test = X.loc[:split_date], X.loc[split_date:]
y_train, y_test = y.loc[:split_date], y.loc[split_date:]

len(X_train), len(X_test)

(1814, 2025)

### 6.1 Logistic Regression (interpretable baseline)

Logistic regression is a strong baseline for noisy financial classification tasks.

We standardize features inside a `Pipeline` to prevent leakage.

In [9]:
logit = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED))
])

logit.fit(X_train, y_train)
proba = logit.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

print(f"Logistic Regression - Test Accuracy: {acc:.3f}")
print(f"Logistic Regression - Test ROC AUC:  {auc:.3f}")

Logistic Regression - Test Accuracy: 0.556
Logistic Regression - Test ROC AUC:  0.500


### 6.2 Random Forest (non-linear baseline)

Tree ensembles can capture non-linear interactions. They can also overfit easily, so we keep settings conservative.

In [10]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=100,
    random_state=SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)

acc_rf = accuracy_score(y_test, pred_rf)
auc_rf = roc_auc_score(y_test, proba_rf)

print(f"Random Forest - Test Accuracy: {acc_rf:.3f}")
print(f"Random Forest - Test ROC AUC:  {auc_rf:.3f}")

Random Forest - Test Accuracy: 0.553
Random Forest - Test ROC AUC:  0.502


## 7. From predictions to a trading strategy

A model is only useful if it improves **risk-adjusted** performance.

A simple rule:
- Position = +1 (long) if predicted probability > threshold
- Otherwise position = 0 (cash)

We also include a small transaction cost per trade to keep things honest.

In [11]:
def prob_to_strategy_returns(
    probs: pd.Series,
    returns: pd.Series,
    threshold: float = 0.55,
    cost_bps: float = 2.0,
) -> pd.Series:
    """Convert predicted probabilities into strategy returns with simple costs.

    cost_bps is applied when position changes (round-trip simplified to one-way per change).
    """
    pos = (probs > threshold).astype(int)
    pos = pos.shift(1).fillna(0)  # trade on next day
    strat = pos * returns

    # transaction cost when changing position
    trades = pos.diff().abs().fillna(0)
    cost = (cost_bps / 10_000) * trades
    return strat - cost

probs_logit = pd.Series(proba, index=X_test.index, name="p_up")
probs_rf = pd.Series(proba_rf, index=X_test.index, name="p_up")

strat_logit = prob_to_strategy_returns(probs_logit, df_ml.loc[X_test.index, "ret"])
strat_rf = prob_to_strategy_returns(probs_rf, df_ml.loc[X_test.index, "ret"])

print_performance("ML Strategy (Logit, test period)", strat_logit)
print_performance("ML Strategy (RF, test period)", strat_rf)


ML Strategy (Logit, test period)
--------------------------------
CAGR:           6.57%
Sharpe:          0.45
Max drawdown: -34.62%
Days:            2025

ML Strategy (RF, test period)
-----------------------------
CAGR:           9.32%
Sharpe:          0.60
Max drawdown: -35.43%
Days:            2025


## 8. Walk-forward validation (more realistic)

A single train/test split can be misleading. Walk-forward validation simulates how models are updated through time:
- train on a window
- test on the next block
- roll forward

We will do this for logistic regression to show the workflow.

In [12]:
tscv = TimeSeriesSplit(n_splits=6)

walk_probs = pd.Series(index=X.index, dtype=float)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr = y.iloc[train_idx]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED))
    ])
    pipe.fit(X_tr, y_tr)
    walk_probs.iloc[test_idx] = pipe.predict_proba(X_te)[:, 1]

walk_probs = walk_probs.dropna()
walk_strat = prob_to_strategy_returns(walk_probs, df_ml.loc[walk_probs.index, "ret"], threshold=0.55, cost_bps=2.0)

print_performance("Walk-forward ML Strategy (Logit)", walk_strat)


Walk-forward ML Strategy (Logit)
--------------------------------
CAGR:           4.13%
Sharpe:          0.35
Max drawdown: -31.03%
Days:            3288


## 9. Deep learning for sequences (LSTM)

Classic ML uses point-in-time features. Deep learning can ingest a **sequence** of past features to predict the next move.

Below we build:
- a sliding window dataset
- an LSTM classifier in PyTorch

To keep the notebook portable, we keep the model small and train for a few epochs.

In [ ]:
# %pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [15]:
class SequenceDataset(Dataset):
    def __init__(self, X: pd.DataFrame, y: pd.Series, seq_len: int = 20):
        self.X = X.values.astype(np.float32)
        self.y = y.values.astype(np.int64)
        self.seq_len = seq_len

    def __len__(self) -> int:
        return len(self.X) - self.seq_len

    def __getitem__(self, idx: int):
        x_seq = self.X[idx:idx + self.seq_len]
        y_next = self.y[idx + self.seq_len]
        return torch.from_numpy(x_seq), torch.tensor(y_next)

class LSTMClassifier(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 32, num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last)

### 9.1 Prepare sequential dataset

We standardize features using **train-only** statistics, then build sequences.

We'll reuse the same date split to keep comparisons consistent.

In [16]:
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 20

# Fit scaler on train only
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=feature_cols)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=feature_cols)

ds_train = SequenceDataset(X_train_scaled, y_train, seq_len=SEQ_LEN)
ds_test  = SequenceDataset(X_test_scaled, y_test, seq_len=SEQ_LEN)

dl_train = DataLoader(ds_train, batch_size=128, shuffle=False)  # do NOT shuffle time series
dl_test  = DataLoader(ds_test, batch_size=256, shuffle=False)

len(ds_train), len(ds_test)

(1794, 2005)

### 9.2 Train LSTM

We optimize cross-entropy loss.

In [17]:
model = LSTMClassifier(n_features=len(feature_cols), hidden_size=32).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, dl):
    model.train()
    total_loss = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(dl.dataset)

@torch.no_grad()
def predict_proba(model, dl) -> np.ndarray:
    model.eval()
    probs = []
    for xb, _ in dl:
        xb = xb.to(device)
        logits = model(xb)
        p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        probs.append(p)
    return np.concatenate(probs)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    l = train_one_epoch(model, dl_train)
    print(f"Epoch {epoch:02d} | train loss {l:.4f}")

Epoch 01 | train loss 0.6882
Epoch 02 | train loss 0.6862
Epoch 03 | train loss 0.6857
Epoch 04 | train loss 0.6853
Epoch 05 | train loss 0.6849


### 9.3 Evaluate LSTM and backtest

We turn predicted probabilities into a trading strategy, same as before, and compare on the test period.

In [18]:
probs_lstm = predict_proba(model, dl_test)

# Align back to dates: SequenceDataset drops the first SEQ_LEN days
test_index_seq = X_test.index[SEQ_LEN:]  # after seq window
probs_lstm_s = pd.Series(probs_lstm, index=test_index_seq)

pred_lstm = (probs_lstm_s >= 0.5).astype(int)
acc_lstm = accuracy_score(y_test.loc[test_index_seq], pred_lstm)
auc_lstm = roc_auc_score(y_test.loc[test_index_seq], probs_lstm_s)

print(f"LSTM - Test Accuracy: {acc_lstm:.3f}")
print(f"LSTM - Test ROC AUC:  {auc_lstm:.3f}")

strat_lstm = prob_to_strategy_returns(probs_lstm_s, df_ml.loc[test_index_seq, "ret"], threshold=0.55, cost_bps=2.0)
print_performance("DL Strategy (LSTM, test period)", strat_lstm)

LSTM - Test Accuracy: 0.553
LSTM - Test ROC AUC:  0.504

DL Strategy (LSTM, test period)
-------------------------------
CAGR:           8.84%
Sharpe:          0.59
Max drawdown: -29.33%
Days:            2005


## 10. Deep learning for sequences (Transformer)

Transformers are another way to process sequences. Compared with LSTMs, they can model longer contexts and interactions.

We implement a compact **Transformer Encoder** for the same classification task. This is adapted from the original notebook's Transformer snippets, rewritten to be clean and connected.

In [19]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, n_features: int, seq_len: int, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model

        # Project features into model dimension
        self.input_fc = nn.Linear(n_features, d_model)

        # Learnable positional embedding
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        # x: [batch, seq_len, n_features]
        h = self.input_fc(x) + self.pos_embedding
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.head(last)

### 10.1 Train and evaluate Transformer

In [20]:
tfm = TimeSeriesTransformer(n_features=len(feature_cols), seq_len=SEQ_LEN, d_model=32, nhead=4, num_layers=2).to(device)
opt_t = torch.optim.Adam(tfm.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch_t(model, dl):
    model.train()
    total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt_t.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt_t.step()
        total += loss.item() * len(xb)
    return total / len(dl.dataset)

EPOCHS_T = 5
for epoch in range(1, EPOCHS_T + 1):
    l = train_one_epoch_t(tfm, dl_train)
    print(f"Epoch {epoch:02d} | train loss {l:.4f}")

probs_t = predict_proba(tfm, dl_test)
probs_t_s = pd.Series(probs_t, index=test_index_seq)

pred_t = (probs_t_s >= 0.5).astype(int)
acc_t = accuracy_score(y_test.loc[test_index_seq], pred_t)
auc_t = roc_auc_score(y_test.loc[test_index_seq], probs_t_s)

print(f"Transformer - Test Accuracy: {acc_t:.3f}")
print(f"Transformer - Test ROC AUC:  {auc_t:.3f}")

strat_t = prob_to_strategy_returns(probs_t_s, df_ml.loc[test_index_seq, "ret"], threshold=0.55, cost_bps=2.0)
print_performance("DL Strategy (Transformer, test period)", strat_t)

Epoch 01 | train loss 0.7482
Epoch 02 | train loss 0.6982
Epoch 03 | train loss 0.6931
Epoch 04 | train loss 0.6861
Epoch 05 | train loss 0.6846
Transformer - Test Accuracy: 0.528
Transformer - Test ROC AUC:  0.502

DL Strategy (Transformer, test period)
--------------------------------------
CAGR:           9.07%
Sharpe:          0.61
Max drawdown: -29.33%
Days:            2005


## 11. What to look for (interpretation)

In markets, **tiny improvements in prediction metrics often do not translate** into better trading after costs.

Key checks:
- Does the strategy beat buy and hold on Sharpe and drawdown?
- Is performance stable across sub-periods (walk-forward)?
- How sensitive is it to the probability threshold and transaction costs?

Below is a quick sensitivity sweep for the logistic walk-forward probabilities.

In [21]:
thresholds = np.arange(0.50, 0.65, 0.02)
rows = []
for th in thresholds:
    r = prob_to_strategy_returns(walk_probs, df_ml.loc[walk_probs.index, "ret"], threshold=float(th), cost_bps=2.0)
    rows.append({
        "threshold": float(th),
        "sharpe": sharpe_ratio(r),
        "max_dd": max_drawdown((1+r).cumprod()),
        "cagr": ((1+r).cumprod().iloc[-1] ** (252/len(r)) - 1) if len(r) else np.nan
    })

pd.DataFrame(rows).set_index("threshold")

,sharpe,max_dd,cagr
threshold,,,
0.50,0.793834,-0.337173,0.124928
0.52,0.690885,-0.337173,0.103779
0.54,0.478065,-0.332350,0.063908
0.56,0.355092,-0.310671,0.041472
0.58,0.433816,-0.283190,0.048496
0.60,0.314499,-0.273487,0.029298
0.62,0.355111,-0.238053,0.030708
0.64,0.272065,-0.222450,0.019635


## 12. Summary

You now have a connected learning curve:

1. Download clean public market data.
2. Define returns and a prediction target.
3. Build interpretable features.
4. Benchmark with classic strategies.
5. Train classical ML models with time-aware splits.
6. Convert predictions into trading rules and evaluate after costs.
7. Scale up to sequence models (LSTM, Transformer) with PyTorch.
8. Use walk-forward validation and threshold sensitivity to avoid overfitting.

Next ideas:
- Multi-asset features (SPY + TLT + GLD) and cross-asset signals
- Volatility targeting / position sizing
- Regression target (predict returns) instead of direction classification
- Probabilistic calibration and uncertainty-aware sizing
